### Multilayer Perceptron (MLP) Model

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report

# --- Standard Loading & Cleaning (per original) ---
df = pd.read_csv('Traffic_dataset_cleaned.csv')
cols_to_drop = ['CRASH_DATE', 'MOST_SEVERE_INJURY', 'INTERSECTION_RELATED_I']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

target = 'SEVERITY'
num_cols = ['NUM_UNITS', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']
cat_cols = [c for c in df.columns if c != target and c not in num_cols]

# Missing Values
for col in num_cols:
    df[col] = df[col].fillna(df[col].median() if not df[col].isnull().all() else 0)
for col in cat_cols + [target]:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].isnull().all() else 'UNKNOWN')

# Undersampling
df_low = df[df['SEVERITY'] == 'LOW'].sample(n=min(200000, len(df[df['SEVERITY'] == 'LOW'])), random_state=42)
df_balanced = pd.concat([df_low, df[df['SEVERITY'] == 'MEDIUM'], df[df['SEVERITY'] == 'HIGH']], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Encode Target for Neural Network
le = LabelEncoder()
df_balanced[target] = le.fit_transform(df_balanced[target])

# Splitting
train_df, temp_df = train_test_split(df_balanced, test_size=0.30, random_state=42, stratify=df_balanced[target])
val_df, test_df = train_test_split(temp_df, test_size=0.3333, random_state=42, stratify=temp_df[target])

# Feature Scaling/Encoding
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

def transform_features(data, is_train=False):
    if is_train:
        num_part = scaler.fit_transform(data[num_cols])
        cat_part = encoder.fit_transform(data[cat_cols])
    else:
        num_part = scaler.transform(data[num_cols])
        cat_part = encoder.transform(data[cat_cols])
    return np.hstack([num_part, cat_part])

X_train = torch.FloatTensor(transform_features(train_df, is_train=True))
y_train = torch.LongTensor(train_df[target].values)
X_val = torch.FloatTensor(transform_features(val_df))
y_val = torch.LongTensor(val_df[target].values)
X_test = torch.FloatTensor(transform_features(test_df))
y_test = torch.LongTensor(test_df[target].values)

# Create DataLoaders
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=1024, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=1024)

In [2]:
class AdvancedMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(AdvancedMLP, self).__init__()
        self.layers = nn.Sequential(
            # Input Layer
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Hidden Layer 1
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Hidden Layer 2
            nn.Linear(256, 128),
            nn.ReLU(),
            
            # Output Layer
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.layers(x)

# Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AdvancedMLP(X_train.shape[1], len(le.classes_)).to(device)

# Loss and Optimizer
# Using label_smoothing to help the model generalize better across classes
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

In [3]:
epochs = 20
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for vX, vy in val_loader:
            vX, vy = vX.to(device), vy.to(device)
            v_out = model(vX)
            val_loss += criterion(v_out, vy).item()
    
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_traffic_mlp.pth')
        
    print(f"Epoch {epoch+1:02d} | Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

# Final Evaluation on Test Set
model.load_state_dict(torch.load('best_traffic_mlp.pth'))
model.eval()
with torch.no_grad():
    y_pred_probs = model(X_test.to(device))
    y_pred = torch.argmax(y_pred_probs, dim=1).cpu().numpy()

print("\n--- Model Evaluation ---")
print(classification_report(y_test.numpy(), y_pred, target_names=le.classes_))

Epoch 01 | Train Loss: 0.5701 | Val Loss: 0.4992
Epoch 02 | Train Loss: 0.4902 | Val Loss: 0.4739
Epoch 03 | Train Loss: 0.4760 | Val Loss: 0.4727
Epoch 04 | Train Loss: 0.4732 | Val Loss: 0.4729
Epoch 05 | Train Loss: 0.4717 | Val Loss: 0.4729
Epoch 06 | Train Loss: 0.4705 | Val Loss: 0.4721
Epoch 07 | Train Loss: 0.4698 | Val Loss: 0.4725
Epoch 08 | Train Loss: 0.4693 | Val Loss: 0.4726
Epoch 09 | Train Loss: 0.4685 | Val Loss: 0.4728
Epoch 10 | Train Loss: 0.4657 | Val Loss: 0.4725
Epoch 11 | Train Loss: 0.4648 | Val Loss: 0.4726
Epoch 12 | Train Loss: 0.4645 | Val Loss: 0.4726
Epoch 13 | Train Loss: 0.4640 | Val Loss: 0.4727
Epoch 14 | Train Loss: 0.4636 | Val Loss: 0.4727
Epoch 15 | Train Loss: 0.4637 | Val Loss: 0.4727
Epoch 16 | Train Loss: 0.4637 | Val Loss: 0.4727
Epoch 17 | Train Loss: 0.4636 | Val Loss: 0.4727
Epoch 18 | Train Loss: 0.4636 | Val Loss: 0.4727
Epoch 19 | Train Loss: 0.4638 | Val Loss: 0.4727
Epoch 20 | Train Loss: 0.4635 | Val Loss: 0.4727

--- Model Evaluatio